# Public Area Cleanliness Monitoring — Notebook + Production Modules

This notebook is the **orchestration layer** on top of four standalone, production-grade Python
modules (kept as separate `.py` files so they can also be run from the command line / deployed
as-is, not just from a notebook):

| File | Role |
|---|---|
| `convert_dataset.py` | COCO (TACO) → YOLO format conversion, with filename-collision fix, box validation/clamping, and 3 class-mapping modes (`single` / `supercategory` / `all`) |
| `train_litter.py` | Trains YOLOv8 on the converted dataset with augmentations tuned for floor-camera footage, then auto-exports for deployment |
| `export_model.py` | Exports trained weights to ONNX + TorchScript and writes a `deployment_config.json` |
| `cleanliness_detector.py` | The full runtime pipeline: litter detection, person detection, centroid tracking, abandoned-object flagging, spill/dirt heuristics, area-based scoring, and a HUD overlay renderer — runnable standalone on a video file/webcam via its `__main__` block |

The notebook below **imports and calls these modules directly** (not reimplementations), runs the
whole pipeline end-to-end on a small synthetic dataset so you can see every stage actually execute,
and tells you exactly what to swap in for a real deployment.

All four files were verified before this notebook was assembled:
- All four `py_compile` cleanly.
- `convert_dataset.py` was run against a synthetic COCO set with intentionally colliding filenames
  (`batch_1/000000.jpg` vs `batch_2/000000.jpg`) and a degenerate zero-width box — the collision fix
  and box validation both worked correctly, and all three class-mapping modes produced the expected
  class lists.
- `train_litter.py` was run for a real (tiny, 1-epoch) training job on that synthetic dataset —
  training completed, weights were saved, and it auto-triggered `export_model.py`, which produced a
  real `.onnx`, a real `.torchscript` file, and `deployment_config.json`.
- `cleanliness_detector.py`'s `CleanlinessMonitor` was run on a synthetic frame end-to-end
  (detection → tracking → spill/dirt heuristics → scoring → HUD overlay), the "custom model not
  found → fall back to stock YOLO" path was exercised deliberately, and the abandoned-object
  flagging logic was unit-tested directly (flags when no person is nearby after the stationary
  threshold; does not flag when a person is within the owner radius).

### Honest scope note
- Litter detection can genuinely be trained on real data — [TACO](http://tacodataset.org/) is a
  real public dataset of photographed litter. The synthetic dataset used below exists only to prove
  the *pipeline* runs; swap in real TACO (or your own labeled photos) before trusting the model.
- There's no equivalent public dataset for spills/wet floors/dirt, so `cleanliness_detector.py`
  uses classical CV heuristics (specular highlights for wet spots, texture-variance anomalies for
  dirt) that need no training data. They're a reasonable starting point, not a finished product —
  expect to tune the thresholds against your real footage, and consider adding a trained classifier
  later once you've collected labeled images (the previous version of this notebook includes a
  ResNet18 transfer-learning template if you want to go that route).
- "Clean" thresholds per area type live in `cleanliness_detector.AREA_CONFIGS` — that's the config
  you work out **with the client**, not something the model decides.


## 0. Setup

If you're on a system that blocks system-wide pip installs (PEP 668,
`externally-managed-environment` error — common on Debian/Ubuntu), either run this notebook inside
a virtual environment (`python3 -m venv venv && source venv/bin/activate`) or in a hosted notebook
(Colab, Kaggle, JupyterHub), where this isn't an issue. The `--break-system-packages` flag below is
only there so this cell also works on locked-down systems directly.

In [ ]:
# Run once.
!pip install -q --break-system-packages ultralytics opencv-python-headless torch torchvision 2>/dev/null || \
 pip install -q ultralytics opencv-python-headless torch torchvision


In [ ]:
import os
import sys
import json
import time
import shutil
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# The four .py files (convert_dataset.py, train_litter.py, export_model.py,
# cleanliness_detector.py) must sit in the SAME FOLDER as this notebook.
PROJECT_DIR = Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

import convert_dataset
import train_litter
import export_model
import cleanliness_detector

print("Modules loaded from:", PROJECT_DIR)


## 1. Dataset — real TACO, or a synthetic smoke-test set

### Option A — real data (what you'll actually train on)
```bash
git clone https://github.com/pedropro/TACO.git
cd TACO && python3 download.py
```
Then point `COCO_ANN_PATH` / `IMAGES_ROOT` below at `TACO/data/annotations.json` and `TACO/data`.

### Option B — synthetic smoke-test set (used below)
So this notebook runs top-to-bottom without a multi-GB download, this cell builds a tiny fake
COCO dataset — including two images that deliberately collide in filename across "batches"
(`batch_1/000000.jpg` vs `batch_2/000000.jpg`), which is exactly the bug `convert_dataset.py` was
fixed to handle. **Replace this cell with real TACO paths for an actual deployment.**


In [ ]:
def build_fake_taco_dataset(root="fake_taco/data", n_per_batch=3, seed=0):
    root = Path(root)
    for batch in ["batch_1", "batch_2"]:
        (root / batch).mkdir(parents=True, exist_ok=True)

    categories = [
        {"id": 1, "name": "Plastic bottle", "supercategory": "Bottle"},
        {"id": 2, "name": "Cigarette", "supercategory": "Cigarette"},
        {"id": 3, "name": "Aluminium foil", "supercategory": "Can"},
    ]
    images, annotations = [], []
    ann_id = img_id = 0
    rng = np.random.default_rng(seed)

    for batch in ["batch_1", "batch_2"]:
        for i in range(n_per_batch):
            fname = f"{batch}/{i:06d}.jpg"  # collides across batches on purpose
            w, h = 320, 240
            img = rng.integers(60, 100, (h, w, 3), dtype=np.uint8)
            cv2.imwrite(str(root / fname), img)
            images.append({"id": img_id, "file_name": fname, "width": w, "height": h})

            for _ in range(rng.integers(1, 3)):
                x, y = int(rng.integers(0, w - 20)), int(rng.integers(0, h - 20))
                bw, bh = int(rng.integers(10, min(60, w - x))), int(rng.integers(10, min(60, h - y)))
                cat = categories[rng.integers(0, len(categories))]
                annotations.append({"id": ann_id, "image_id": img_id, "category_id": cat["id"],
                                     "bbox": [x, y, bw, bh]})
                ann_id += 1
            img_id += 1

    # one degenerate (zero-width) box, to exercise convert_dataset.py's validation/skip path
    annotations.append({"id": ann_id, "image_id": 0, "category_id": 1, "bbox": [10, 10, 0, 15]})

    coco = {"images": images, "annotations": annotations, "categories": categories}
    ann_path = root / "annotations.json"
    ann_path.write_text(json.dumps(coco))
    return str(ann_path), str(root)

COCO_ANN_PATH, IMAGES_ROOT = build_fake_taco_dataset()
print("Synthetic dataset written:", COCO_ANN_PATH)


### 1.1 Convert to YOLO format via `convert_dataset.py`

In [ ]:
data_yaml_path = convert_dataset.convert_coco_to_yolo(
    coco_ann_path=COCO_ANN_PATH,
    images_root=IMAGES_ROOT,
    out_dir="litter_yolo_dataset",
    class_mode="single",   # 'single' = everything -> one 'litter' class (recommended to start);
                             # 'supercategory' = ~28 classes; 'all' = TACO's 60 original classes
    val_split=0.34,         # small val_split here only because the synthetic set is tiny
    copy_files=True,
)
print("data.yaml written to:", data_yaml_path)


## 2. Train the litter detector via `train_litter.py`

This calls the real training function — augmentations tuned for floor-camera footage
(`flipud`/`fliplr` for orientation invariance, rotation, mosaic, mixup, HSV jitter), automatic
CPU/GPU device selection, best-weights saving, and automatic export to ONNX + TorchScript at the
end via `export_model.py`.

For a real run against full TACO, bump `epochs` up (50+) and use a GPU. The tiny numbers below are
only to keep this notebook fast to execute as a smoke test — the training loop itself is
identical to what a real run does.


In [ ]:
best_model_path = train_litter.train(
    data_yaml=data_yaml_path,
    base_model="yolov8n.pt",   # swap for yolov8s.pt / yolov8m.pt for more accuracy, more compute
    epochs=1,                   # smoke-test value — use 50+ for a real run
    imgsz=320,                  # smoke-test value — use 640 for a real run
    batch=2,
    device="cpu",
    workers=0,
    project_dir="runs_litter",
    exp_name="smoke_test",
    export_deployment=True,     # auto-runs export_model.export_all_formats at the end
    class_mode="single",
)
print("\nBest weights saved to:", best_model_path)


In [ ]:
print("Exported files in models/:")
for f in sorted(Path("models").glob("*")):
    print(" -", f.name)

print("\ndeployment_config.json:")
print(json.dumps(json.loads(Path("models/deployment_config.json").read_text()), indent=2))


## 3. Run the full monitoring pipeline via `cleanliness_detector.py`

`CleanlinessMonitor` combines everything: litter detection (your freshly trained model, or a stock
YOLO fallback if a custom model isn't found yet), person detection (for the abandoned-object
owner-proximity check), centroid tracking, the spill/dirt CV heuristics, area-based scoring, and a
HUD overlay for visualization.


In [ ]:
def make_synthetic_floor_frame():
    '''Stand-in for a real camera frame: replace with cv2.imread(...) on your own footage.'''
    frame = np.full((480, 640, 3), 90, dtype=np.uint8)
    frame = cv2.GaussianBlur(frame, (5, 5), 0)
    cv2.circle(frame, (420, 300), 35, (230, 230, 230), -1)   # simulated wet spot (bright, low-sat)
    cv2.circle(frame, (420, 300), 35, (200, 200, 200), 8)
    cv2.ellipse(frame, (150, 380), (60, 25), 20, 0, 360, (40, 35, 30), -1)  # simulated dirt smear
    noise = np.random.randint(0, 15, frame.shape, dtype=np.uint8)
    return cv2.add(frame, noise)

monitor = cleanliness_detector.CleanlinessMonitor(
    litter_model_path=best_model_path,   # falls back to stock YOLO automatically if this doesn't exist
    person_model_path="yolov8n.pt",
    area_type="corridor",                # see Part 4 for what this controls
)

frame = make_synthetic_floor_frame()
report, annotated = monitor.process_frame(frame, timestamp=time.time())

print(json.dumps(report, indent=2, default=str))

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"{report['status']} — cleanliness {report['clean_score']}/100")
plt.axis("off")
plt.show()


### 3.1 Abandoned-object behaviour over time

A left object only gets flagged once it's been stationary past the area's threshold **and** no
person is within the owner radius. This simulates the same static frame across several timestamps
to show the flag turning on.


In [ ]:
t0 = time.time()
for dt in [0, 5, 20, 40]:
    r, _ = monitor.process_frame(frame, timestamp=t0 + dt)
    print(f"t+{dt:>3}s -> litter={r['details']['litter_count']}, "
          f"abandoned={r['details']['abandoned_count']}, status={r['status']}")


### 3.2 Running it live (video file / webcam)

`cleanliness_detector.py` is also a standalone CLI script — you don't have to go through the
notebook for a real deployment:
```bash
python3 cleanliness_detector.py \
    --litter-model models/litter_detector_best.pt \
    --person-model yolov8n.pt \
    --area corridor \
    --source floor_camera.mp4      # or 0 for a live webcam
```
It opens an OpenCV window with the live HUD (area, score, status, litter count, spill score) and
draws boxes for spills (cyan), dirt (orange), persons (blue), and litter/abandoned items
(yellow / red). Press `q` to quit.


## 4. Define "clean" per area type

This lives in `cleanliness_detector.AREA_CONFIGS` — a plain config, not a model decision. Agree
these numbers with the client per zone type (a restroom tolerates far less than a warehouse).


In [ ]:
for name, cfg in cleanliness_detector.AREA_CONFIGS.items():
    print(f"{name:10s} -> max_litter={cfg.max_litter_items}, max_abandoned={cfg.max_abandoned_objects}, "
          f"spill_thresh={cfg.spill_score_threshold}, dirt_thresh={cfg.dirt_score_threshold}, "
          f"weights(litter/spill/dirt)=({cfg.weight_litter}, {cfg.weight_spill}, {cfg.weight_dirt})")


To add or change an area type, edit `AREA_CONFIGS` directly in `cleanliness_detector.py`
(or monkey-patch it from a notebook for experimentation, as below) — then re-run
`process_frame` and compare scores.


In [ ]:
from cleanliness_detector import AreaCleanlinessConfig

# Example: add a stricter "food_court" area type and compare its verdict against the same frame
cleanliness_detector.AREA_CONFIGS["food_court"] = AreaCleanlinessConfig(
    "food_court", max_litter_items=0, max_abandoned_objects=0,
    spill_score_threshold=0.05, dirt_score_threshold=0.08,
)

for area in ["warehouse", "corridor", "food_court"]:
    r = cleanliness_detector.score_cleanliness(
        area_type=area,
        litter_count=report["details"]["litter_count"],
        abandoned_count=report["details"]["abandoned_count"],
        spill_score=report["details"]["spill_score"],
        dirt_score=report["details"]["dirt_score"],
    )
    print(f"{area:12s} -> score={r['clean_score']:>5} status={r['status']}")


## 5. Next steps for a real deployment

1. **Swap the synthetic dataset for real TACO** (or your own site photos) in Part 1, and retrain
   with real `epochs`/`imgsz`/`batch` in Part 2 — ideally on a GPU.
2. **Collect a small labeled set of real floor images** (clean / dirty / wet) from your actual
   cameras to validate — and eventually improve on — the spill/dirt heuristics' thresholds.
3. **Calibrate a floor mask per fixed camera position** and pass it into `CleanlinessMonitor`
   (`floor_mask=...`) so the heuristics ignore walls, furniture, and reflective glass.
4. **Agree `AREA_CONFIGS` thresholds with the client on-site**, and expect to revisit them after
   the first few weeks of real alerts.
5. **Track precision/recall on real footage**, not just training metrics — nuisance false
   positives will kill adoption faster than an occasional miss.
6. **Add a human-in-the-loop review step** early on: flagged frames get a quick operator
   thumbs-up/down, and disagreements feed back into the training data.
